<a href="https://colab.research.google.com/github/tamk283/Entity-Client-DDNet/blob/main/comfyui_colab_Flux.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Git clone the [ComfyUI](https://github.com/comfyanonymous/ComfyUI) repo and install the requirements.

In [ ]:
!git clone -q https://github.com/comfyanonymous/ComfyUI
%pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu130
%pip install -q -r ./ComfyUI/requirements.txt

In [ ]:
# Some Helpful Custom Nodes
!git clone -q https://github.com/ltdrdata/ComfyUI-Manager /content/ComfyUI/custom_nodes/comfyui-manager

!git clone -q https://github.com/ClownsharkBatwing/RES4LYF.git /content/ComfyUI/custom_nodes/RES4LYF
!git clone -q https://github.com/rgthree/rgthree-comfy.git /content/ComfyUI/custom_nodes/rgthree-comfy
!git clone -q https://github.com/yolain/ComfyUI-Easy-Use.git /content/ComfyUI/custom_nodes/ComfyUI-Easy-Use

Download the vae and dual text encoder needed for FLUX - (Compulsory)

In [ ]:
!wget -q --show-progress https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors -P /content/ComfyUI/models/text_encoders/
!wget -q --show-progress https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors -P /content/ComfyUI/models/text_encoders/
!wget -q --show-progress https://huggingface.co/cocktailpeanut/xulf-dev/resolve/main/ae.safetensors -P /content/ComfyUI/models/vae/

# 8-step distilled lora for fast inference on dev models
# !wget -q --show-progress https://huggingface.co/alimama-creative/FLUX.1-Turbo-Alpha/resolve/main/diffusion_pytorch_model.safetensors -P /content/ComfyUI/models/loras


The **FP8 version** runs smoothly on the free tier (without LoRA), but if you want to use it with a LoRA, it requires more resources than the free Colab tier provides.  
In that case, you can opt for the [gguf](https://huggingface.co/collections/QuantStack/flux-ggufs-68264cfc663d50c418940b30) quantized versions or the [Nunchaku](https://huggingface.co/nunchaku-tech/models) versions instead.

>**Note** : You need only one , either a gguf quantized model (under 10GB) or the Nunchanku model

---


### Option 1 : fp8 quantized - from [Comfy-Org](https://huggingface.co/Comfy-Org/flux1-dev/tree/main)

In [ ]:
# fp8 - needs more than free tier
# !wget -q --show-progress https://huggingface.co/Comfy-Org/flux1-dev/resolve/main/flux1-dev-fp8.safetensors -P /content/ComfyUI/models/checkpoints

### Option 2 : [**gguf**](https://huggingface.co/city96/FLUX.1-dev-gguf) models.  

Git clone the [ComfyUI-GGUF](https://github.com/city96/ComfyUI-GGUF) repo and follow the instructions.

In [ ]:
# %pip install --upgrade gguf
# !rm -rf ./custom_nodes/ComfyUI-GGUF
# !git clone https://github.com/city96/ComfyUI-GGUF /content/ComfyUI/custom_nodes/ComfyUI-GGUF

In [ ]:
# Download the one you need

# flux1-dev
# !wget -q --show-progress https://huggingface.co/city96/FLUX.1-dev-gguf/resolve/main/flux1-dev-Q6_K.gguf -P /content/ComfyUI/models/unet

# flux1-kontext-dev
# !wget -q --show-progress https://huggingface.co/QuantStack/FLUX.1-Kontext-dev-GGUF/resolve/main/flux1-kontext-dev-Q6_K.gguf -P /content/ComfyUI/models/unet

# flux1-krea-dev
# !wget -q --show-progress https://huggingface.co/QuantStack/FLUX.1-Krea-dev-GGUF/resolve/main/flux1-krea-dev-Q6_K.gguf -P /content/ComfyUI/models/unet

# flux1-schnell
# !wget -q --show-progress https://huggingface.co/city96/FLUX.1-schnell-gguf/resolve/main/flux1-schnell-Q6_K.gguf -P /content/ComfyUI/models/unet

### Option 3 : [**Nunchaku**](https://nunchaku.tech/docs/ComfyUI-nunchaku/workflows/t2i.html#nunchaku-flux-1-dev-json) models - recommended

In [ ]:
import torch
print(torch.__version__)


In [ ]:
!git clone -q https://github.com/mit-han-lab/ComfyUI-nunchaku /content/ComfyUI/custom_nodes/nunchaku_nodes
!pip install -q https://github.com/nunchaku-tech/nunchaku/releases/download/v1.0.0/nunchaku-1.0.0+torch2.8-cp312-cp312-linux_x86_64.whl

In [ ]:
# flux1-dev
!wget -q --show-progress https://huggingface.co/nunchaku-tech/nunchaku-flux.1-dev/resolve/main/svdq-int4_r32-flux.1-dev.safetensors -P /content/ComfyUI/models/diffusion_models

#flux1-krea-dev
# !wget -q --show-progress https://huggingface.co/nunchaku-tech/nunchaku-flux.1-krea-dev/resolve/main/svdq-int4_r32-flux.1-krea-dev.safetensors -P /content/ComfyUI/models/diffusion_models

#flux1-schnell
# !wget -q --show-progress https://huggingface.co/nunchaku-tech/nunchaku-flux.1-schnell/resolve/main/svdq-int4_r32-flux.1-schnell.safetensors -P /content/ComfyUI/models/diffusion_models

#flux1-kontext-dev
# !wget -q --show-progress https://huggingface.co/nunchaku-tech/nunchaku-flux.1-kontext-dev/resolve/main/svdq-int4_r32-flux.1-kontext-dev.safetensors -P /content/ComfyUI/models/diffusion_models

### Run ComfyUI with ngrok

In [ ]:
!pip install -q pyngrok

from pyngrok import ngrok
import subprocess, socket, time
from google.colab import userdata

NGROK_TOKEN = userdata.get("NGROK_TOKEN")

# Set ngrok token
!ngrok config add-authtoken $NGROK_TOKEN

# Start ComfyUI in background
subprocess.Popen(["python", "/content/ComfyUI/main.py", "--dont-print-server"])

# Wait until port is open
port = 8188
while True:
    try:
        sock = socket.create_connection(("127.0.0.1", port), timeout=2)
        sock.close()
        print("✅ ComfyUI server is running on port", port)
        break
    except OSError:
        print("⏳ Waiting for ComfyUI to start...")
        time.sleep(2)

# Start ngrok tunnel
public_url = ngrok.connect(8188, bind_tls=True)
print("🌐 Public URL:", public_url)


### Run ComfyUI with localtunnel

In [19]:
import subprocess
import threading
import time
import socket
import os

# 1. Скачиваем cloudflared (без токенов)
print("Скачиваем cloudflared...")
subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"])
subprocess.run(["dpkg", "-i", "cloudflared-linux-amd64.deb"])

# 2. Функция для запуска туннеля
def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            sock.close()
            break
        sock.close()

    print("\nComfyUI загружен, запускаем туннель...\n")
    p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com" in l:
            print("ССЫЛКА ДЛЯ ВХОДА:", l[l.find("http"):], end='')

# 3. Запускаем
threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
print("Запускаем ComfyUI в фоновом режиме...")
subprocess.Popen(["python3", "/content/ComfyUI/main.py", "--dont-print-server"])

Скачиваем cloudflared...
Запускаем ComfyUI в фоновом режиме...


<Popen: returncode: None args: ['python3', '/content/ComfyUI/main.py', '--do...>

In [20]:
import subprocess
# Проверим, запущен ли процесс ComfyUI и cloudflared
ps = subprocess.run(["ps", "aux"], capture_output=True, text=True)
print("=== Запущенные процессы ===")
for line in ps.stdout.splitlines():
    if "python" in line or "cloudflared" in line:
        print(line)


=== Запущенные процессы ===
root          65  0.4  0.0      0     0 ?        Z    14:06   0:15 [python3] <defunct>
root          66  0.1  0.6 105364 83300 ?        S    14:06   0:03 python3 /usr/local/bin/colab-fileshim.py
root         132  0.2  1.2 630688 164232 ?       Sl   14:06   0:07 /usr/bin/python3 /usr/local/bin/jupyter-server --debug --transport="ipc" --ip=172.28.0.12 --ServerApp.token= --port=9000 --FileContentsManager.root_dir=/ --FileContentsManager.allow_hidden=True --ServerApp.log_format="|%(levelname)s|%(message)s" --ServerApp.iopub_data_rate_limit=1e10 --MappingKernelManager.root_dir=/content
root         401  0.5  1.0 1109276 135848 ?      Ssl  14:07   0:17 /usr/bin/python3 -m colab_kernel_launcher -f /root/.local/share/jupyter/runtime/kernel-3f276541-e6b9-4f25-ae9e-5ae66b926b4c.json
root        3410  0.0  0.0      0     0 ?        Z    14:17   0:00 [python3] <defunct>
root       16157  6.7  8.8 9886096 1174128 ?     Sl   14:56   0:10 python3 /content/ComfyUI/main.py -

In [21]:
# Проверим, слушает ли сервер ComfyUI порт 8188
import socket
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
result = sock.connect_ex(('127.0.0.1', 8188))
if result == 0:
    print("\n[УСПЕХ] Сервер ComfyUI успешно запущен на порту 8188!")
else:
    print("\n[ОШИБКА] Сервер ComfyUI еще не запустился или упал с ошибкой.")
sock.close()


[УСПЕХ] Сервер ComfyUI успешно запущен на порту 8188!


In [23]:
import time
import subprocess
import re

print("Получаем ссылку для входа...")
time.sleep(2)

try:
    print("Создаем новую ссылку для подключения к уже запущенному ComfyUI...")
    p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    found = False
    for i in range(150):
        line = p.stderr.readline().decode('utf-8')
        if "trycloudflare.com" in line:
            # Используем регулярное выражение для извлечения точного URL
            match = re.search(r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com', line)
            if match:
                link = match.group(0)
                print("\n=== ВАША ССЫЛКА ДЛЯ ВХОДА ===")
                print(link)
                print("=============================")
                found = True
                break
        time.sleep(0.1)
    if not found:
        print("Не удалось быстро перехватить лог. Пожалуйста, запустите эту ячейку еще раз через несколько секунд.")
except Exception as e:
    print("Ошибка при получении ссылки:", e)

Получаем ссылку для входа...
Создаем новую ссылку для подключения к уже запущенному ComfyUI...

=== ВАША ССЫЛКА ДЛЯ ВХОДА ===
https://polymer-steel-flame-explain.trycloudflare.com


In [ ]:
# !npm install -g localtunnel

In [ ]:
# import subprocess
# import threading
# import time
# import socket
# import urllib.request
# import sys

# def wait_for_port(port):
#     while True:
#         time.sleep(0.5)
#         sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
#         if sock.connect_ex(('127.0.0.1', port)) == 0:
#             sock.close()
#             break
#         sock.close()

#     print("\nComfyUI finished loading, launching localtunnel...\n")
#     print("Your public IP is:",
#           urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

#     p = subprocess.Popen(["lt", "--port", str(port)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
#     for line in iter(p.stdout.readline, ''):
#         sys.stdout.write(line)
#         sys.stdout.flush()

# threading.Thread(target=wait_for_port, args=(8188,), daemon=True).start()

# subprocess.run(["python", "/content/ComfyUI/main.py"])
